# MonsoonLens AI: Historical Rainfall Pattern Analysis
## Notebook 01: Exploratory Data Analysis & Quality Verification

**Objective:**
Explore the cleaned and structured historical rainfall datasets generated by the preprocessing pipeline.
Inspect distributions, geographical variability across 36 subdivisions, seasonal rainfall dynamics, and historical trends (1901–2015).


In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    CLEANED_WIDE_PATH,
    MONTHLY_LONG_PATH,
    ANNUAL_PATH,
    SEASONAL_PATH,
    QUALITY_FLAGS_PATH,
    METADATA_PATH
)

# Plot styling configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11


### 1. Ingest Processed Datasets

In [ ]:
df_wide = pd.read_csv(CLEANED_WIDE_PATH)
df_long = pd.read_csv(MONTHLY_LONG_PATH)
df_annual = pd.read_csv(ANNUAL_PATH)
df_seasonal = pd.read_csv(SEASONAL_PATH)
df_flags = pd.read_csv(QUALITY_FLAGS_PATH)

with open(METADATA_PATH, 'r') as f:
    metadata = json.load(f)

print(f"Cleaned Wide:    {df_wide.shape}")
print(f"Monthly Long:    {df_long.shape}")
print(f"Annual Series:   {df_annual.shape}")
print(f"Seasonal Series: {df_seasonal.shape}")
print(f"Quality Flags:   {df_flags.shape}")


### 2. Missing Value Analysis by Subdivision

In [ ]:
# Missing records summary from quality flags
missing_flags = df_flags[df_flags['flag_type'].str.contains('MISSING')]
print(f"Total missing flags: {len(missing_flags)}")

missing_by_sub = missing_flags.groupby('subdivision')['field'].count().sort_values(ascending=False)
print("\nMissing observations by subdivision:")
print(missing_by_sub)


### 3. Annual Rainfall Statistics & Distribution

In [ ]:
annual_stats = df_annual.groupby('SUBDIVISION')['ANNUAL_VALIDATED'].agg(
    mean='mean',
    std='std',
    min='min',
    median='median',
    max='max'
).sort_values(by='mean', ascending=False)

print("Top 5 Wettest Subdivisions (Annual Mean mm):")
print(annual_stats.head())

print("\nTop 5 Driest Subdivisions (Annual Mean mm):")
print(annual_stats.tail())


### 4. Seasonal Rainfall Proportions Across Subdivisions

In [ ]:
season_summary = df_seasonal.groupby('SEASON')['RAINFALL_VALIDATED'].mean().reset_index()
season_summary = season_summary.sort_values(by='RAINFALL_VALIDATED', ascending=False)

print("National Mean Rainfall by Season (mm):")
print(season_summary)


### 5. Historical All-India Annual Rainfall Trend (1901–2015)

In [ ]:
all_india_annual = df_annual.groupby('YEAR')['ANNUAL_VALIDATED'].mean()

plt.figure(figsize=(14, 5))
plt.plot(all_india_annual.index, all_india_annual.values, color='#1f77b4', lw=1.5, label='National Mean Annual Rainfall')
plt.plot(all_india_annual.rolling(10, min_periods=5).mean(), color='#d62728', lw=2.5, label='10-Year Moving Average')
plt.axhline(all_india_annual.mean(), color='black', ls='--', lw=1, label=f'115-Year Mean ({all_india_annual.mean():.1f} mm)')

plt.title('All-India Historical Mean Annual Rainfall (1901–2015)', fontsize=14, weight='bold')
plt.xlabel('Year')
plt.ylabel('Rainfall (mm)')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()
